### 1. Using Pdb

The python debugger is a pretty good tool you can use to debug your application.

with pdb, you can debug in 2 main ways -
1. post-mortem debugging: debug after entire application has executed.. Ig this would be useful when you're completely clueless about where is the source of your bug located
2. interactive debugging: Add line `Import pdb; pdb.set_trace()` to where you're suspicious of code to set a breakpoint. The code then reaches this part during execution and opens the interactive debugger. You can then explore using the interactive commands..

i. l (list)
ii. n (next)
iii. c (continue)
iv. s (step)
v. r (return)
vi. b (break)
vii. Python

### i. Interactive Debugging

In [ ]:
from urllib.request import Request, urlopen
import ssl
import pdb

def myFunction():
    myssl = ssl.create_default_context()
    myssl.check_hostname = False
    myssl.verify_mode = ssl.CERT_NONE

    pdb.set_trace()  # execution will pause here
    req = Request('https://tutorialedge.net', headers={'User-Agent': 'Mozilla/5.0'})
    response = urlopen(req, context=myssl)
    print("Response status:", response.status)

myFunction()

# NOTE: have a look at all of the commands you can run while being in the interactive debugger. Most of them are listed in the above cell..

### ii. Postmortem Debugging

Nothing was said about Postmortem debugging sadly... I think i'll have to research that myself... It seemed useful hence..

### 2. Catching exceptions in child threads

Apparently the way to do this is to communicate those exceptions / errors to the parent thread

In [ ]:
import sys
import threading
import time
import queue

def myThread(q):
    while True:
        try:
            time.sleep(2)
            raise Exception(
                f"Exception Thrown In Child Thread {threading.current_thread()}. The parent is {threading.main_thread()}"
            )
        except:
            q.put(sys.exc_info())
        finally:
            print("exception put in queue")
            break

q = queue.Queue()
t = threading.Thread(name = "child", target=myThread, args=(q,), daemon=True)
t.start()

if __name__ == '__main__':
    while True:
        try:
            exception = q.get(block=False)
            print("found an exception in queue")
        except queue.Empty:
            pass
            print("nothing found")
        else:
            print(exception)
            print("done!")
            break
        finally: 
            time.sleep(2)

I'm starting to understand what this particular section of catching exceptions in child threads taught us..

It taught us that exceptions don't cross threads... If a child faces an exception, it'll pass silently in that thread alone.. It won't reach the parent thread, even when you might want the parent thread to know about it.

This is something that needs to be handled on our end... We need to maintain some communication, that relays exceptions to parent, which can then decide what to do about it.

`q.put(sys.exc_info())`

This is the key line to propagate such errors across threads - by passing it in a shared data structure that must be thread-safe, like a queue.